In [51]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
from locallib.query import get_users, Query, get_surveys
from locallib.picarrodb import *
from locallib.box import *
from datetime import datetime, date, timedelta
import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils.datetime import from_excel, to_excel
from openpyxl.pivot import fields as xl_pivot_fields


In [53]:
os.chdir('/home/sandbox/personal-repos/DA-3564')

In [54]:
BOX_FOLDER_ID = 372723239821

In [55]:
a = Query(get_users(customer_name = 'Cadent', user_table = '#TempUsers'))
a.set_child(Query(get_surveys(user_table = '#TempUsers', start_date = '2026-01-01')))
surveys = a.execute(EU2_Conn)

In [56]:
surveys.db.set_query('SELECT ReportId, SurveyId FROM ReportDrivingSurvey JOIN Report ON ReportDrivingSurvey.ReportId = Report.Id WHERE SurveyId IN (SELECT SurveyId FROM #TempSurveys)')
df = surveys.db.execute(EU2_Conn, source_col = 'SurveyId', temp_table_name = '#TempSurveys', erase_table = True)

/home/sandbox/personal-repos/locallib_packages/locallib/picarrodb/PConnection.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(self.query, temp_conn)


In [57]:
surveys_wreport = pd.merge(surveys, df, on='SurveyId', how='left').sort_values(by='StartDateTimeSurvey')
surveys_wreport['Date'] = surveys_wreport.StartDateTimeSurvey.dt.date
surveys_wreport['Hour'] = surveys_wreport['StartDateTimeSurvey'].dt.hour
for index, row in surveys_wreport.iterrows():
    if (row['Hour'] >= 0) & (row['Hour'] <= 8):
        surveys_wreport.at[index, 'AnalysisDate'] = row['Date'] - timedelta(days=1)
    else:
        surveys_wreport.at[index, 'AnalysisDate'] = row['Date']

In [58]:
surveys_wreport[surveys_wreport['Date'] == date(2026, 3, 24)][['Hour', 'Date', 'AnalysisDate']]

,Hour,Date,AnalysisDate
81,0,2026-03-24,2026-03-23
82,0,2026-03-24,2026-03-23
83,0,2026-03-24,2026-03-23
84,0,2026-03-24,2026-03-23
85,0,2026-03-24,2026-03-23
...,...,...,...
15,23,2026-03-24,2026-03-24
16,23,2026-03-24,2026-03-24
17,23,2026-03-24,2026-03-24
18,23,2026-03-24,2026-03-24


In [59]:
excel_path = 'CADENT_SurveyTracker.xlsx'

# Load existing workbook (do not overwrite existing sheets)
wb = load_workbook(excel_path)

# Remove sheet named 'RawData' if it exists, to overwrite its content
if 'RawData' in wb.sheetnames:
    std = wb['RawData']
    wb.remove(std)

# Create new 'RawData' sheet
ws = wb.create_sheet('RawData')

# Write DataFrame to the 'RawData' sheet
for r_idx, row in enumerate(dataframe_to_rows(surveys_wreport, index=False, header=True), 1):
    for c_idx, value in enumerate(row, 1):
        ws.cell(row=r_idx, column=c_idx, value=value)

ws_pivot = wb['Dashboard']
pivot = ws_pivot._pivots[0]
wb.save(excel_path)
box_folder = BoxFile(excel_path, BOX_FOLDER_ID)
box_folder.upload()
